In [ ]:
# <editor-fold desc="Imports">
import os
import time
import torch
import warnings
from tqdm import tqdm

import nibabel as nib
import pandas as pd
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import Dataset, DataLoader
# </editor-fold>

In [ ]:

# <editor-fold desc="Clinical Data Preparation and Cleaning">

# Default dataframe cleaning
clinical_data_path = "/scratch/tgoedietdoebe/AI-project/data/labels_BinClass.csv"
clinical_data = pd.read_csv(clinical_data_path)
clinical_data.dropna(inplace=True)
clinical_data.reset_index(drop=True, inplace=True)
clinical_data['w8_responder'] = clinical_data['w8_responder'].map({'Yes': 1, 'No': 0})
clinical_data['Stage1TX'] = clinical_data['Stage1TX'].map({'SER': 1, 'PLA': 0})
subjects_to_remove = ['CU0058', 'CU0059', 'CU0060', 'CU0061', 'CU0068', 'CU0074']
filtered_clinical_data = clinical_data[~clinical_data['ProjectSpecificId'].isin(subjects_to_remove)]
filtered_clinical_data.reset_index(drop=True, inplace=True)
sertraline_df = filtered_clinical_data[filtered_clinical_data['Stage1TX'] == 1]


fmri_data_path = "/data/projects/depredict/repositories/EMBARC/data/data_bids/derivatives/_fmriprep/output/"
subject_ids = sertraline_df['ProjectSpecificId'].tolist()
fmri_files = {}
missing_subjects = []
for sub_id in subject_ids:
    file_pattern = os.path.join(fmri_data_path,
                                f"sub-{sub_id}/ses-1/func/sub-{sub_id}_ses-1_task-rest1_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz")
    if os.path.exists(file_pattern):
        fmri_files[sub_id] = file_pattern
    else:
        missing_subjects.append(sub_id)
processed_df = sertraline_df[sertraline_df['ProjectSpecificId'].isin(fmri_files.keys())]
processed_df = processed_df.copy()
processed_df['fMRI_path'] = processed_df['ProjectSpecificId'].map(fmri_files)
processed_df
# </editor-fold>

In [ ]:

# <editor-fold desc="Data Classes & Functions">
class fMRIDataset(Dataset):
    def __init__(self, df, data_dict, transform=None):
        """
        df: DataFrame containing at least 'ProjectSpecificId' and 'w8_responder'
        data_dict: {subject_id: torch.Tensor} holding the precomputed mean images
        transform: any optional transform
        """
        self.data = df.reset_index(drop=True)
        self.data_dict = data_dict
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        subject_id = row['ProjectSpecificId']
        label = row['w8_responder']

        # Retrieve the preloaded mean image
        mean_image = self.data_dict[subject_id]  # Shape: [1, D, H, W]

        # Optional transform
        if self.transform:
            mean_image = self.transform(mean_image)

        return mean_image, torch.tensor(label, dtype=torch.float32)

def preload_mean_images(df, mean_image_path):
    """
    Loads all precomputed mean .pt files specified in df['ProjectSpecificId'] into memory (RAM)
    and returns a dictionary: {subject_id: mean_image_tensor, ...}.
    """
    data_dict = {}
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Preloading mean images"):
        subject_id = row["ProjectSpecificId"]
        pt_path = os.path.join(mean_image_path, f"{subject_id}_mean.pt")

        if not os.path.exists(pt_path):
            print(f"Warning: {pt_path} not found. Skipping subject {subject_id}.")
            continue

        # Load the precomputed mean image
        mean_image = torch.load(pt_path)
        data_dict[subject_id] = mean_image

    return data_dict

# def preload_data(df, preprocessed_path):
#     """
#     Loads all .pt files specified in df['ProjectSpecificId'] into memory (RAM)
#     and returns a dictionary: {subject_id: fmri_tensor, ...}.
#
#     Each fmri_tensor is the full 4D data for that subject: [T, D, H, W] or [T, 1, D, H, W].
#     """
#     data_dict = {}
#     # Use tqdm for a nice progress bar
#     for i, row in tqdm(df.iterrows(), total=len(df), desc="Preloading data"):
#         subject_id = row["ProjectSpecificId"]
#         pt_path = os.path.join(preprocessed_path, f"{subject_id}.pt")
#
#         if not os.path.exists(pt_path):
#             # Handle missing files, skip or raise error
#             print(f"Warning: {pt_path} not found. Skipping subject {subject_id}.")
#             continue
#
#         # Load the preprocessed fMRI tensor
#         start_load = time.time()
#         fmri_tensor = torch.load(pt_path)
#         load_time = time.time() - start_load
#         print(f"Loading time for {subject_id}: {load_time:.3f}s")
#
#         data_dict[subject_id] = fmri_tensor
#
#     return data_dict

def preprocess_and_save(fmri_path, subject_id, save_dir="/scratch/tgoedietdoebe/AI-project/data/processed/"):
    try:
        print(f"Processing {subject_id}...")
        fmri_img = nib.load(fmri_path)
        fmri_data = fmri_img.get_fdata()  # Should be [97, 115, 97, 180] ideally

        if fmri_data.ndim != 4:
            print(f"❌ Skipping {subject_id}: Expected 4D fMRI data, got shape {fmri_data.shape}")
            return

        if fmri_data.shape[3] != 180:
            print(f" Warning: {subject_id} has unexpected timepoints: {fmri_data.shape}")

        # Convert to PyTorch tensor (reorder to [T, D, H, W])
        # fmri_tensor = torch.tensor(fmri_data).permute(3, 0, 1, 2) # 1.6GB
        fmri_tensor = torch.tensor(fmri_data, dtype=torch.float32).permute(3, 0, 1, 2)

        # Save
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{subject_id}.pt")
        torch.save(fmri_tensor, save_path)

        print(f"✅ Saved {subject_id} | Final tensor shape: {fmri_tensor.shape} | Path: {save_path}")

    except Exception as e:
        print(f"❌ Failed to preprocess {subject_id}: {e}")
# </editor-fold>


In [ ]:
# # Load 4D fMRI tensors
# data_dict = preload_data(processed_df, preprocessed_path="/scratch/tgoedietdoebe/AI-project/data/processed/")
#
# # Precompute and save mean images
# precompute_mean_images(data_dict, save_dir="/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/")

In [ ]:

# <editor-fold desc="Training & Evaluation Loop">

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    warnings.filterwarnings("ignore", category=FutureWarning)

    history = {
        'step': [],
        'epoch': [],
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': []
    }

    total_start_time = time.time()

    for epoch in range(num_epochs):
        print(f"\n🚀 Starting Epoch {epoch + 1}/{num_epochs}")
        model.train()
        total_loss = 0
        total_data_time = 0
        total_gpu_time = 0
        start_epoch = time.time()
        batch_losses = []
        step_counter = 0
        eval_interval = 5

        for batch_idx, (images, labels) in tqdm(enumerate(train_loader), total=len(train_loader), desc=f'Train Epoch {epoch+1}'):
            step_counter += 1

            # start_data = time.time()
            images, labels = images.to(device), labels.to(device)
            # data_time = time.time() - start_data
            # print(f"Data loading time: {data_time:.3f}s")

            start_gpu = time.time()
            optimizer.zero_grad()
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            gpu_time = time.time() - start_gpu

            # total_data_time += data_time
            total_gpu_time += gpu_time
            total_loss += loss.item()
            batch_losses.append(loss.item())


            # Evaluate the model and capture metrics
            if step_counter % eval_interval == 0:
                val_loss, val_accuracy, val_precision, val_recall, val_f1 = evaluate_model(model, val_loader, criterion)

                # Store metrics
                history['step'].append(batch_idx + 1)
                history['epoch'].append(epoch + 1)
                history['train_loss'].append(total_loss / (batch_idx + 1))
                history['val_loss'].append(val_loss)
                history['val_accuracy'].append(val_accuracy)
                history['val_precision'].append(val_precision)
                history['val_recall'].append(val_recall)
                history['val_f1'].append(val_f1)

                # Log metrics
                print(f"\nStep {batch_idx + 1}:")
                print(f"   Train Loss: {total_loss / (batch_idx + 1):.4f}")
                print(f"   Val Loss: {val_loss:.4f} | Accuracy: {val_accuracy * 100:.2f}%")
                print(f"   Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1 Score: {val_f1:.4f}")

    # Record the end time of training
    total_end_time = time.time()
    total_training_time = total_end_time - total_start_time

    print(f"\n⏱️ Total Training Time: {total_training_time:.2f} seconds")

    return history

def evaluate_model(model, val_loader, criterion):
    model.eval()
    total_loss = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Evaluating", leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels.float())
            total_loss += loss.item()

            # Collect predictions and labels
            predicted = (torch.sigmoid(outputs) > 0.5).int()
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_val_loss = total_loss / len(val_loader)
    accuracy = (np.array(all_predictions) == np.array(all_labels)).mean()
    precision = precision_score(all_labels, all_predictions)
    recall = recall_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions)

    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Accuracy: {accuracy * 100:.2f}%")
    print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")

    return avg_val_loss, accuracy, precision, recall, f1
# </editor-fold>


In [ ]:

# <editor-fold desc="Model Definition">
class Simple3DCNN(nn.Module):
    def __init__(self):
        super(Simple3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 16, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1)

        # Use Global Average Pooling to reduce feature map size efficiently
        self.global_pool = nn.AdaptiveAvgPool3d((4, 4, 4))

        # FC Layer adjusted for the reduced size
        self.fc1 = nn.Linear(64 * 4 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        # Reduce feature map size while retaining granularity
        x = self.global_pool(x)

        x = x.view(x.size(0), -1)  # Flatten for FC layer
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class HighAccuracy3DCNN(nn.Module):
    def __init__(self):
        super(HighAccuracy3DCNN, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=7, stride=1, padding=3),
            nn.BatchNorm3d(8),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv2 = nn.Sequential(
            nn.Conv3d(8, 16, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv3 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.conv4 = nn.Sequential(
            nn.Conv3d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )

        self.feature_dim = 64 * 6 * 7 * 6

        self.dropout = nn.Dropout(p=0.5)
        self.fc1 = nn.Linear(self.feature_dim, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv1(x)  # [B, 8, ~48, ~57, ~48]
        x = self.conv2(x)  # [B, 16, ~24, ~28, ~24]
        x = self.conv3(x)  # [B, 32, ~12, ~14, ~12]
        x = self.conv4(x)  # [B, 64, ~6, ~7, ~6]

        x = x.view(x.size(0), -1)  # Flatten
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)  # No sigmoid here — use BCEWithLogitsLoss

        return x
# </editor-fold>


In [ ]:
# def precompute_mean_images(data_dict, save_dir="/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/"):
#     os.makedirs(save_dir, exist_ok=True)
#     for subject_id, fmri_tensor in tqdm(data_dict.items(), desc="Precomputing mean images"):
#         # Compute the mean image
#         mean_image = fmri_tensor.mean(dim=0)  # Shape: [D, H, W]
#
#         # Normalize the mean image
#         eps = 1e-6
#         mean_image = (mean_image - mean_image.mean()) / (mean_image.std() + eps)
#
#         # Add a channel dimension to match model input shape
#         mean_image = mean_image.unsqueeze(0)  # Shape: [1, D, H, W]
#
#         # Save the mean image
#         save_path = os.path.join(save_dir, f"{subject_id}_mean.pt")
#         torch.save(mean_image, save_path)
#         print(f"Saved mean image for {subject_id} at {save_path}")

In [ ]:

#<editor-fold desc="Non-Slurm Execution">

# Preload mean images
mean_image_path = "/home/tgoedietdoebe/scratch/AI-project/data/processed/mean_image/T_180/"
data_dict = preload_mean_images(processed_df, mean_image_path)

# Prepare datasets and data loaders
train_data, test_data = train_test_split(processed_df, test_size=0.2)
train_data, val_data = train_test_split(train_data, test_size=0.25)

train_dataset = fMRIDataset(train_data, data_dict=data_dict)
val_dataset = fMRIDataset(val_data, data_dict=data_dict)
test_dataset = fMRIDataset(test_data, data_dict=data_dict)

batch_size = 1 # 16 * 7
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers= 16, pin_memory= True )
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Model Preperation
model = HighAccuracy3DCNN()
# model = Simple3DCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Current Device: {torch.cuda.current_device()}")
print(f"GPU: {torch.cuda.get_device_name(torch.cuda.current_device())}")
# </editor-fold>


In [ ]:
len(train_loader)  # Number of batches in the training set

In [ ]:
# Training & Evaluation
history = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=2)

In [ ]:

# <editor-fold desc="Epoch Plots">

# Plot training and validation loss
plt.plot(history['epoch'], history['train_loss'], label='Training Loss')
plt.plot(history['epoch'], history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.show()

# Plot precision, recall, and F1 score
plt.plot(history['epoch'], history['val_precision'], label='Precision')
plt.plot(history['epoch'], history['val_recall'], label='Recall')
plt.plot(history['epoch'], history['val_f1'], label='F1 Score')
plt.xlabel('Epoch')
plt.ylabel('Score')
plt.title('Precision, Recall, and F1 Score')
plt.legend()
plt.show()

# Combined metrics plot
plt.figure(figsize=(10, 6))
plt.plot(history['epoch'], history['train_loss'], label='Training Loss')
plt.plot(history['epoch'], history['val_loss'], label='Validation Loss')
plt.plot(history['epoch'], history['val_accuracy'], label='Validation Accuracy')
plt.plot(history['epoch'], history['val_precision'], label='Precision')
plt.plot(history['epoch'], history['val_recall'], label='Recall')
plt.plot(history['epoch'], history['val_f1'], label='F1 Score')
plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('Training and Validation Metrics')
plt.legend()
plt.show()
# </editor-fold>

In [ ]:
# Plot training loss over steps
plt.figure(figsize=(10, 6))
plt.plot(history['step'], history['train_loss'], label='Training Loss', marker='o')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss Over Steps')
plt.legend()
plt.grid()
plt.show()

# Plot validation metrics over steps
plt.figure(figsize=(10, 6))
plt.plot(history['step'], history['val_loss'], label='Validation Loss', marker='o')
plt.plot(history['step'], history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.plot(history['step'], history['val_precision'], label='Precision', marker='o')
plt.plot(history['step'], history['val_recall'], label='Recall', marker='o')
plt.plot(history['step'], history['val_f1'], label='F1 Score', marker='o')
plt.xlabel('Step')
plt.ylabel('Metrics')
plt.title('Validation Metrics Over Steps')
plt.legend()
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot training loss on the primary y-axis
ax1.plot(history['step'], history['train_loss'], label='Training Loss', color='blue', marker='o')
ax1.set_xlabel('Step')
ax1.set_ylabel('Training Loss', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Plot validation accuracy on the secondary y-axis
ax2 = ax1.twinx()
ax2.plot(history['step'], history['val_accuracy'], label='Validation Accuracy', color='green', marker='o')
ax2.set_ylabel('Validation Accuracy', color='green')
ax2.tick_params(axis='y', labelcolor='green')

# Add title and grid
plt.title('Training Loss and Validation Accuracy Over Steps')
fig.tight_layout()
plt.grid()
plt.show()